# 02 – Cleaning and temporal split

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilaria-cesa/Txt_lab/blob/main/notebooks/02_cleaning_and_splitting.ipynb)

Work in progress. Starting from the table built in notebook 01, this notebook:

1. removes the papers whose abstract is **exactly** the same as another one (keeps the earliest);
2. lists the papers that share a **title** and applies the manual review (`drop` column);
3. adds the technique / domain labels (`src/label_mapping.py`);
4. splits the data **by year**: train = 2015–2024, validation = 2025, test = 2026 (until the download date);
5. (optional) rebalances the **train set only** by reducing the papers that only have the generic label;
6. saves the splits in Google Drive.

Everything that is still to be decided is marked `TODO`.

## 1. Setup and parameters

In [ ]:
import os, sys, subprocess, pathlib, json
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/Txt_lab"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "https://github.com/ilaria-cesa/Txt_lab", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull"], check=True)      # picks up files pushed after the first clone
    SRC = f"{REPO}/src"
    DATA_DIR = "/content/drive/MyDrive/Txt_lab_data"
else:
    SRC = str(pathlib.Path("..") / "src")
    DATA_DIR = "../data"
sys.path.insert(0, SRC)

CSV_PATH = f"{DATA_DIR}/arxiv_ai_abstracts.csv"
OUT_DIR = f"{DATA_DIR}/cleaning_splitting"
os.makedirs(OUT_DIR, exist_ok=True)

import label_mapping as lm, duplicates as dp, splitting as sp

In [ ]:
# --- Parameters ----------------------------------------------------------------------------
MERGE_CATEGORIES = False    # TODO: True if copies of the same abstract carry different categories (see section 2)
APPLY_TITLE_REVIEW = False  # set to True after filling the `drop` column of title_duplicates_review.csv (section 3)
TRAIN_END_YEAR, VAL_YEAR, TEST_YEAR = 2024, 2025, 2026
GENERAL_CAP = None          # TODO: max number of train papers whose only label is General ML/AI; None = no rebalancing
SEED = 42

In [ ]:
df = pd.read_csv(CSV_PATH, dtype={"arxiv_id": str}, usecols=dp.REVIEW_COLS + ["primary_category"])
print(df.shape)
log = {"rows_start": len(df)}

## 2. Identical abstracts

`exact_abstract_duplicates.csv` lists every involved paper. Things to look at: groups whose copies have **different categories**
(then use `MERGE_CATEGORIES = True`), groups with more than two copies and very short abstracts (placeholders).

In [ ]:
exact = dp.report_exact_abstract_duplicates(df, f"{OUT_DIR}/exact_abstract_duplicates.csv")
df1, n_exact = dp.drop_exact_abstract_duplicates(df, merge_categories=MERGE_CATEGORIES)
log["removed_exact_abstract"] = n_exact
print(f"{len(df)} -> {len(df1)} papers ({n_exact} removed, earliest copy kept)")

## 3. Same title: manual review

Open `title_duplicates_review.csv` (in Google Drive) with Google Sheets or Excel. Groups with `status = REVIEW` are the ones to look at;
`auto` groups (almost identical abstracts) already have `drop = 1` on the later copies; `different` groups are probably different papers
with a generic title and are kept. Write `1` in the `drop` column of the rows to remove, save, then set `APPLY_TITLE_REVIEW = True`.

In [ ]:
REVIEW_CSV = f"{OUT_DIR}/title_duplicates_review.csv"
review = dp.build_title_review(df1, REVIEW_CSV)

In [ ]:
if APPLY_TITLE_REVIEW:
    df2, n_title = dp.apply_title_decisions(df1, REVIEW_CSV)
    print(f"{len(df1)} -> {len(df2)} papers ({n_title} removed after the manual review)")
else:
    df2, n_title = df1, 0
    print("Title review NOT applied yet: set APPLY_TITLE_REVIEW = True after editing the CSV.")
log["removed_title_review"] = n_title
log["rows_clean"] = len(df2)

## 4. Labels

Technique labels (classification target) and domain labels (reference for the topic model) from `src/label_mapping.py`.

In [ ]:
lm.check_coverage()
df2 = lm.add_labels(df2)
print(lm.label_report(df2))

## 5. Temporal split by year

Train = years up to `TRAIN_END_YEAR`, validation = `VAL_YEAR`, test = `TEST_YEAR`. Whole calendar years, so no paper of the test
period can be seen in training. Look at how the label proportions change from one split to the next: this is the temporal drift
that the models will have to face.

In [ ]:
splits = sp.split_by_year(df2, TRAIN_END_YEAR, VAL_YEAR, TEST_YEAR)
sp.split_summary(splits)

## 6. Rebalancing (train only)

The papers whose only technique label is `General ML/AI` are far more numerous than the others. With `GENERAL_CAP` they are reduced
**in the training set only**, sampling uniformly within each month so that the temporal composition does not change. Validation and
test keep their natural distribution. Choose the cap after reading the table of section 5 (suggestion: about the size of the largest
specific class of the training set).

In [ ]:
train = splits["train"]
if GENERAL_CAP is None:
    train_bal = None
    print("No rebalancing yet (GENERAL_CAP = None).")
else:
    train_bal = sp.cap_general(train, GENERAL_CAP, SEED)
    print(f"train: {len(train)} -> {len(train_bal)} papers")
    display(sp.split_summary({"train": train, "train_balanced": train_bal}))

## 7. Save

In [ ]:
for name, d in splits.items():
    d.to_csv(f"{OUT_DIR}/{name}.csv.gz", index=False)
if train_bal is not None:
    train_bal.to_csv(f"{OUT_DIR}/train_balanced.csv.gz", index=False)
log.update({k: len(v) for k, v in splits.items()})
log.update({"params": {"train_end": TRAIN_END_YEAR, "val_year": VAL_YEAR, "test_year": TEST_YEAR,
                       "general_cap": GENERAL_CAP, "seed": SEED, "merge_categories": MERGE_CATEGORIES}})
pathlib.Path(f"{OUT_DIR}/cleaning_log.json").write_text(json.dumps(log, indent=2))
print(json.dumps(log, indent=2))

## Notes

- The splits are saved in Google Drive (`data/` is not versioned in git).
- `categories` and `n_categories` contain the label: **never use them as features**.
- Only the training set is ever rebalanced; hyperparameters and decision thresholds are chosen on the validation set, and the test set is used once.